# Gladius · Roman cutouts on Nexus

<div style="background:#21142d;color:#f8f3ff;border-radius:14px;padding:24px 28px;margin:20px 0">
<p style="color:#c9a4ed;margin:0 0 8px;font-size:12px;letter-spacing:2px">REAL MAST DATA</p>
<p style="font-size:23px;margin:0 0 8px">One calibrated image, native cutouts, retained metadata.</p>
<p style="color:#d7c6e8;margin:0">Supply a local .env path. Keep the token and observation outputs in your own environment.</p>
</div>

This notebook queries Roman through MAST and makes cutouts from one actual
calibrated detector image. It has **no synthetic-data fallback**. It is separate
from the documentation site and is not executed by Read the Docs.

Install from an authorized checkout with `python -m pip install -e '.[roman,mast,notebook]'`
and select the matching Python kernel. Run the cells in order. Restricted images,
pointing information and parent metadata remain restricted in the derived outputs.
The tracked notebook intentionally has no saved results.

## 1 · Configure your session

**Required:** set `ENV_FILE` to an existing file containing `MAST_API_TOKEN=...`.
Do not paste the token into a notebook cell. This explicitly selected file supplies
the token for the session; there is no search of other directories.

Choose a writable scratch workspace with enough space for a full detector image.
The parent is staged temporarily during extraction; the cutouts and run records
remain in `WORKSPACE/gladius-mast-test`.

`TARGET_CSV=None` uses the selected image's archive centre. That is a convenient
sky coordinate, **not a source identification**. Supply your own CSV if you have
targets known to fall on the selected image. CSV columns are `id,ra,dec` in ICRS
degrees, or `id,x,y` in zero-based parent pixels; extra columns are retained.

In [ ]:
from pathlib import Path

ENV_FILE = None  # REQUIRED: e.g. Path("/your/private/location/.env")
WORKSPACE = Path.cwd() / "gladius-example"
TARGET_CSV = None  # Optional Path to your own CSV of targets on the chosen image.
CUTOUT_SIZE = 150  # A 150 × 150 array of native detector pixels.
MAX_PARENT_MIB = 512  # Reject larger advertised files before download.
WCS_ROUNDTRIP_TOLERANCE = 0.01  # Pixels; explicit notebook choice, package default is 0.001.

In [ ]:
import os
from dotenv import dotenv_values

if ENV_FILE is None:
    raise ValueError("Set ENV_FILE to the path of your local credential file, then rerun this cell.")
ENV_FILE = Path(ENV_FILE).expanduser().resolve()
if not ENV_FILE.is_file():
    raise FileNotFoundError("ENV_FILE must point to an existing file; no archive request was made.")
credentials = dotenv_values(ENV_FILE, interpolate=False)
if not (credentials.get("MAST_API_TOKEN") or "").strip():
    raise ValueError("The selected file must contain a nonempty MAST_API_TOKEN.")
os.environ["MAST_API_TOKEN"] = credentials["MAST_API_TOKEN"]
del credentials

WORKSPACE = Path(WORKSPACE).expanduser().resolve()
if not isinstance(CUTOUT_SIZE, int) or isinstance(CUTOUT_SIZE, bool) or CUTOUT_SIZE < 1:
    raise ValueError("CUTOUT_SIZE must be a positive integer.")
if not isinstance(MAX_PARENT_MIB, (int, float)) or MAX_PARENT_MIB <= 0:
    raise ValueError("MAX_PARENT_MIB must be positive.")
print("Credential file loaded. Token is not displayed.")

## 2 · Select one real calibrated image

This uses the same metadata-only selection as the GUI's **Load test**: sample up
to five Level 2 datasets and select the smallest known-size calibrated product
among them. It does not claim to find the archive's smallest image. No image is
downloaded in this cell.

Inspect the filename, advertised size and target positions before running the
extraction cell. The size check uses archive metadata, not a streaming byte quota.
An inaccessible archive or an empty calibrated sample raises an error instead of
substituting fabricated data.

In [ ]:
import json
from html import escape

import numpy as np
from IPython.display import HTML, display

from gladius_roman import RunConfig, read_targets_csv, run
from gladius_roman.inputs import mast_test_configuration
from gladius_roman.io import read_cutout

def show_table(headers, rows):
    heading = "".join(f"<th style='text-align:left;padding:10px'>{escape(str(h))}</th>" for h in headers)
    body = "".join(
        "<tr>" + "".join(f"<td style='padding:10px;border-top:1px solid #b29acb'>{escape(str(v))}</td>" for v in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table style='width:100%;border-collapse:collapse'><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

selection = mast_test_configuration(WORKSPACE)
advertised_mib = selection["download_bytes"] / 1024**2
show_table(["Dataset", "Calibrated parent", "Advertised MiB"], [
    [selection["dataset"], selection["filename"], f"{advertised_mib:.1f}"]
])
if advertised_mib > MAX_PARENT_MIB:
    raise ValueError("Selected parent exceeds MAX_PARENT_MIB. No image was downloaded; choose a deliberate larger limit or a different sample.")

settings = selection["configuration"]
settings["extraction"].update(size=CUTOUT_SIZE, no_overlap="skip")
settings["astrometry"]["roundtrip_tolerance_pixels"] = WCS_ROUNDTRIP_TOLERANCE
settings["storage"]["formats"] = ["asdf", "fits"]
settings["stacks"]["enabled"] = False  # One image does not need a display stack.
if TARGET_CSV is not None:
    settings["targets"] = [target.model_dump(mode="json") for target in read_targets_csv(TARGET_CSV)]
config = RunConfig.model_validate(settings).resolved(WORKSPACE)
show_table(["Target", "Coordinates", "Metadata fields"], [
    [t.id, f"{t.ra:.7f}, {t.dec:.7f} deg" if t.ra is not None else f"{t.x}, {t.y} parent pixels", ", ".join(t.metadata) or "—"]
    for t in config.targets
])

## 3 · Make the cutouts

**This cell downloads the selected full parent image.** Gladius opens that image
once, processes all targets, and removes only its own staged download afterward.
Existing local inputs are never deleted. Outputs retain SCI/ERR/DQ and parent
metadata. The numerical WCS check can reject an inconsistent transform; it cannot
establish absolute sky accuracy. This notebook explicitly uses a 0.01-pixel
round-trip tolerance (the package default is 0.001). A real sampled parent had
about 0.0034 pixels of numerical inverse error; this is why the notebook makes
the tolerance visible in the configuration cell. This does not correct or
certify the supplied sky solution.

The output manifest determines which products completed. Non-overlapping targets
are skipped in this example. If none overlap, choose a matching target/image
sample rather than treating an empty result as success.

In [ ]:
import yaml

WORKSPACE.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "example-config.yaml").write_text(
    yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False), encoding="utf-8"
)
manifest_path = run(config, progress=print)
run_directory = manifest_path.parent
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
entries = list(manifest["entries"].values())
completed = [entry for entry in entries if entry["status"] == "complete"]
show_table(["Run status", "Completed", "Skipped", "Failed"], [[
    manifest["status"], len(completed),
    sum(e["status"] == "skipped" for e in entries),
    sum(e["status"] == "failed" for e in entries),
]])
if manifest["status"] != "complete":
    raise RuntimeError("Run failed. Inspect manifest['errors'] locally before continuing.")
if not completed:
    raise RuntimeError("No targets overlapped this image. Supply a matching target CSV/image sample.")
print("Manifest:", manifest_path.relative_to(WORKSPACE))

## 4 · Inspect the native image planes

Each row below is one cutout: **SCI**, **ERR**, and the fraction of pixels carrying
each of the first two DQ bits. NaN padding remains NaN; padded DQ pixels carry
DO_NOT_USE. The plotted science stretch is for display only and does not alter
the arrays. At most four targets are plotted to keep notebook output manageable.

In [ ]:
import matplotlib.pyplot as plt
from astropy.visualization import AsinhStretch, ImageNormalize, PercentileInterval

cutouts = []
for entry in completed[:4]:
    filename = next(name for name in entry["files"] if name.endswith(".asdf"))
    cutouts.append((entry, read_cutout(run_directory / filename)))

with plt.rc_context({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False}):
    fig, axes = plt.subplots(len(cutouts), 2, figsize=(10, 4.2 * len(cutouts)), squeeze=False, constrained_layout=True)
    fig.set_facecolor("#fbf9fd")
    for row, (entry, cutout) in enumerate(cutouts):
        for col, (plane, label) in enumerate((("data", "SCI"), ("err", "ERR"))):
            values = cutout.arrays[plane]
            finite = values[np.isfinite(values)]
            if not finite.size:
                raise ValueError("Cutout has no finite values to display; inspect overlap and DQ.")
            norm = ImageNormalize(finite, interval=PercentileInterval(99), stretch=AsinhStretch())
            ax = axes[row, col]
            im = ax.imshow(values, origin="lower", cmap="magma", norm=norm, interpolation="nearest")
            ax.set_title(f"{entry['target_id']} · {label}", loc="left", color="#62398b", fontweight="bold")
            ax.set_xlabel("Cutout x (pixel)")
            ax.set_ylabel("Cutout y (pixel)")
            fig.colorbar(im, ax=ax, fraction=.046, pad=.04, label=entry.get("data_units") or "Native units")
    plt.show()

show_table(["Target", "Shape", "DO_NOT_USE", "SATURATED"], [
    [entry["target_id"], str(cutout.arrays["data"].shape),
     f"{np.mean((cutout.arrays['dq'] & 1) != 0):.1%}",
     f"{np.mean((cutout.arrays['dq'] & 2) != 0):.1%}"]
    for entry, cutout in cutouts
])

## 5 · Check the metadata and exact sky transform

The original parent footprint is provenance, not the cutout's own footprint.
Gladius keeps the parent tree under `parent_metadata`; derived coverage is stored
separately in `cutout_metadata`. The exact translated WCS is available through
`read_cutout` for both ASDF and Gladius FITS products.

Here we compare ASDF/FITS science arrays and inspect only selected metadata.
This is an output consistency check, not external astrometric calibration.

In [ ]:
import asdf
from gladius_roman.core import pixel_to_world

entry, cutout = cutouts[0]
asdf_name = next(name for name in entry["files"] if name.endswith(".asdf"))
fits_name = next(name for name in entry["files"] if name.endswith(".fits"))
fits_cutout = read_cutout(run_directory / fits_name)
for plane in ("data", "err", "dq"):
    np.testing.assert_array_equal(cutout.arrays[plane], fits_cutout.arrays[plane])

with asdf.open(run_directory / asdf_name, lazy_load=False, memmap=False) as af:
    derived = dict(af.tree["cutout_metadata"])

ny, nx = cutout.arrays["data"].shape
ra, dec = pixel_to_world(cutout.wcs, (nx - 1) / 2, (ny - 1) / 2)
show_table(["Property", "Value"], [
    ["ASDF / FITS native planes", "Exactly equal"],
    ["Original parent metadata sections", ", ".join(cutout.parent_metadata)],
    ["Pixel mapping", json.dumps(cutout.mapping, default=str)],
    ["Centre from translated WCS (ICRS deg)", f"{float(ra):.8f}, {float(dec):.8f}"],
    ["Cutout valid footprint", derived.get("s_region")],
    ["Virtual grid footprint", derived.get("virtual_s_region")],
])

## 6 · Repeat the run

Resume is enabled by default. With matching configuration, code/dependency
versions, input identities and output checksums, completed cutouts are reused.
Missing or corrupt products are made again. MAST discovery and parent staging can
still occur, so this cell may **download the same parent again**; resume is not a
network cache. Run it only if you want to demonstrate that behaviour.

In [ ]:
REPEAT_RUN = False

if REPEAT_RUN:
    repeated_path = run(config, progress=print)
    repeated = json.loads(repeated_path.read_text(encoding="utf-8"))
    assert repeated_path == manifest_path
    assert repeated["status"] == "complete"
    print("Same run identity; see progress messages for reused cutouts.")
else:
    print("Repeat skipped. Set REPEAT_RUN=True to try resume.")

## Keep or share the right things

- Native products, the resolved YAML and manifest are in your selected workspace.
- ASDF/FITS preserve parent and target metadata, including pointing information.
  Keep results private while their inputs are restricted.
- The YAML contains configuration, not your MAST token. Keep the `.env` private.
- Before sharing or committing this notebook, **clear every cell output**.
  Do not use plots or metadata from this run as public examples until the data
  themselves are cleared for release.

For batch use, reuse the saved YAML with `gladius run example-config.yaml` in an
environment with your token available. In the CLI, `.env` is loaded beside that
YAML; the explicit `ENV_FILE` path above is a notebook setting.